In [1]:
import boto3
from pyspark.sql import SparkSession
import os

In [23]:
MINIO_ACCESS_KEY= "minioadmin"
MINIO_SECRET_KEY= "minioadmin"
MINIO_ENDPOINT = "http://minio:9000"
BUCKET_NAME= "datalake"
LOCAL_DATA_PATH= "/raw_mount"

In [24]:
for (root, dirs, files) in os.walk(LOCAL_DATA_PATH):
    print("root: ", root)
    print("dirs: ", dirs)
    print("files: ", files)

root:  /raw_mount
dirs:  ['clinvar', 'dgidb', 'ncbi', 'opentargets']
files:  []
root:  /raw_mount/clinvar
dirs:  []
files:  ['variant_summary.txt.gz']
root:  /raw_mount/dgidb
dirs:  []
files:  ['interactions.tsv']
root:  /raw_mount/ncbi
dirs:  []
files:  ['gene_info.gz']
root:  /raw_mount/opentargets
dirs:  ['association_by_datasource_direct']
files:  []
root:  /raw_mount/opentargets/association_by_datasource_direct
dirs:  []
files:  ['part-00000-0009005d-4472-49b8-9b32-378935bacdd2-c000.snappy.parquet', 'part-00001-0009005d-4472-49b8-9b32-378935bacdd2-c000.snappy.parquet', 'part-00002-0009005d-4472-49b8-9b32-378935bacdd2-c000.snappy.parquet', 'part-00003-0009005d-4472-49b8-9b32-378935bacdd2-c000.snappy.parquet', 'part-00004-0009005d-4472-49b8-9b32-378935bacdd2-c000.snappy.parquet', 'part-00005-0009005d-4472-49b8-9b32-378935bacdd2-c000.snappy.parquet', 'part-00006-0009005d-4472-49b8-9b32-378935bacdd2-c000.snappy.parquet', 'part-00007-0009005d-4472-49b8-9b32-378935bacdd2-c000.snappy.par

In [6]:
def create_bucket(bucket_name):
    s3_client= boto3.resource("s3",
                              endpoint_url= MINIO_ENDPOINT,
                              aws_access_key_id= MINIO_ACCESS_KEY,
                              aws_secret_access_key= MINIO_SECRET_KEY,
                              config=boto3.session.Config(signature_version='s3v4'))
    if s3_client.Bucket(bucket_name) not in s3_client.buckets.all():
        s3_client.create_bucket(Bucket=bucket_name)
        print(f"Created bucket: {bucket_name}")

In [19]:
def upload_raw_to_minio(local_dir, bucket):
    s3_client= boto3.client("s3",
                            endpoint_url= MINIO_ENDPOINT,
                            aws_access_key_id= MINIO_ACCESS_KEY,
                            aws_secret_access_key= MINIO_SECRET_KEY,
                            config=boto3.session.Config(signature_version='s3v4'))
    if not os.path.exists(local_dir):
        print(f"Error: Local data path '{local_dir}' not found in container. Check volume mounts.")
        return


    for root, _, files in os.walk(local_dir):
        for file in files:
            local_file_path= os.path.join(root, file)
            rel_file_path= os.path.relpath(local_file_path, local_dir)
            object_key = f"raw/{rel_file_path}"
            # print(local_file_path)
            # print(object_key)
            try:
                s3_client.upload_file(local_file_path, bucket, object_key)
                print(f"Uploaded: {local_file_path}")
            except:
                print(f"Failed to upload: {local_file_path}")
                
# upload_raw_to_minio(LOCAL_DATA_PATH, "datalake")

In [20]:
# create_bucket(BUCKET_NAME)
upload_raw_to_minio(LOCAL_DATA_PATH, BUCKET_NAME)

Uploaded: /raw_mount/opentargets/association_by_datasource_direct/part-00000-0009005d-4472-49b8-9b32-378935bacdd2-c000.snappy.parquet
Uploaded: /raw_mount/opentargets/association_by_datasource_direct/part-00001-0009005d-4472-49b8-9b32-378935bacdd2-c000.snappy.parquet
Uploaded: /raw_mount/opentargets/association_by_datasource_direct/part-00002-0009005d-4472-49b8-9b32-378935bacdd2-c000.snappy.parquet
Uploaded: /raw_mount/opentargets/association_by_datasource_direct/part-00003-0009005d-4472-49b8-9b32-378935bacdd2-c000.snappy.parquet
Uploaded: /raw_mount/opentargets/association_by_datasource_direct/part-00004-0009005d-4472-49b8-9b32-378935bacdd2-c000.snappy.parquet
Uploaded: /raw_mount/opentargets/association_by_datasource_direct/part-00005-0009005d-4472-49b8-9b32-378935bacdd2-c000.snappy.parquet
Uploaded: /raw_mount/opentargets/association_by_datasource_direct/part-00006-0009005d-4472-49b8-9b32-378935bacdd2-c000.snappy.parquet
Uploaded: /raw_mount/opentargets/association_by_datasource_dir